<a href="https://colab.research.google.com/github/JonasFanZ/114-2-Programing-Language/blob/main/%E3%80%8CHW1_%E6%97%A5%E5%B8%B8%E6%94%AF%E5%87%BA%E9%80%9F%E7%AE%97%E8%88%87%E5%88%86%E6%94%A4_ipynb%E3%80%8D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#日常支出速算與分攤（作業一）
- 目標：從 Sheet 讀「消費紀錄」→ 計總額/分類小計/AA 分攤 → 寫回 Sheet Summary 分頁。
- AI 點子（可選）：請模型總結本週花錢習慣與建議（例如「外食過多」）。
- Sheet 欄位：date, category, item, amount, payer

GoogleSheet: https://docs.google.com/spreadsheets/d/1V4jupaF0rdVdEDwVR6QghG1pNb1bQFStDaN9zC4C2X4/edit?usp=sharing

In [293]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()

gc = gspread.authorize(creds)

In [294]:
import pandas as pd
# read data and put it in a dataframe
# 在 google 工作表載入 gsheets
gsheets = gc.open_by_url('https://docs.google.com/spreadsheets/d/1V4jupaF0rdVdEDwVR6QghG1pNb1bQFStDaN9zC4C2X4/edit?usp=sharing')

In [295]:
# 從 gsheets 的 All-whiteboard-device 載入 sheets
sheets = gsheets.worksheet('工作表1').get_all_values()
# 將 sheets1 資料載入 pd 的 DataFrame 進行分析
df = pd.DataFrame(sheets[1:], columns=sheets[0])
# 取得最前面的5筆資料
df.head()

,日期,分類,品項,金額,付款人,備註
0,2026-01-01,飲食,午餐,120,范子昊,0
1,2026-02-03,交通,公車,12,范子昊,0
2,2026-03-01,飲食,早餐,100,范子昊,0


In [296]:
import datetime

In [297]:
# 讓使用者輸入資料
date_str = input("請輸入日期 (格式：YYYY-MM-DD): ")
# 檢查日期格式是否正確
datetime.datetime.strptime(date_str, '%Y-%m-%d')
category = input("請輸入分類: ")
item = input("請輸入品項: ")
amount = float(input("請輸入金額: "))
payer = input("請輸入付款人: ")
notes = input("請輸入備註: ")

請輸入日期 (格式：YYYY-MM-DD): 2026-03-07
請輸入分類: 飲食
請輸入品項: 午餐
請輸入金額: 120
請輸入付款人: 范子昊
請輸入備註: 0


In [298]:
new_data = pd.DataFrame([{
    '日期': date_str,
    '分類': category,
    '品項': item,
    '金額': amount,
    '付款人': payer,
    '備註': notes
}])

# 使用 concat() 將新資料合併到舊的 df 中
df = pd.concat([df, new_data], ignore_index=True)
df

,日期,分類,品項,金額,付款人,備註
0,2026-01-01,飲食,午餐,120,范子昊,0
1,2026-02-03,交通,公車,12,范子昊,0
2,2026-03-01,飲食,早餐,100,范子昊,0
3,2026-03-07,飲食,午餐,120.0,范子昊,0


In [299]:
# 第一步：取得工作表物件
worksheet = gsheets.worksheet('工作表1')

# 確保 '日期' 欄位是 datetime 類型
df['日期'] = pd.to_datetime(df['日期'])

# 第二步：對 DataFrame 進行排序 (df 已經包含了最新資料，並且 '日期' 已轉為 datetime)
df_sorted = df.sort_values(by='日期', ascending=True)

# 第三步：準備要寫入 Google Sheet 的資料
# 獲取原始的欄位名稱 (不包含 '月份' 等分析用途的欄位)
# 根據原始 Colab 筆記，Sheet 欄位是 date, category, item, amount, payer, 備註
original_columns = ['日期', '分類', '品項', '金額', '付款人', '備註']

# 篩選並重新排列 df_sorted 的列，以符合 Google Sheet 的欄位順序，並明確創建一個副本
df_to_write = df_sorted[original_columns].copy()

# 將 '日期' 欄位轉換為字串格式，以避免 JSON 序列化錯誤
df_to_write['日期'] = df_to_write['日期'].dt.strftime('%Y-%m-%d')

# 確保 '金額' 欄位是數字，並處理潛在的非數字值和空值
df_to_write['金額'] = pd.to_numeric(df_to_write['金額'], errors='coerce').fillna(0).astype(int)

# 將 DataFrame 轉換成列表的列表 (list of lists)
# 並在轉換之前，將 NaN 值替換為空字串，以避免 JSON 格式錯誤
# 同時，在最前面加入欄位名稱作為第一行
data_to_write = [df_to_write.columns.values.tolist()] + df_to_write.fillna('').values.tolist()

# 第四步：清除工作表1的所有內容 (確保是空白的，以便寫入完整排序的資料)
worksheet.clear()

# 第五步：將排序後的資料寫入工作表1
# 使用 update 方法，從 A1 開始寫入，這會覆蓋整個範圍
worksheet.update(values=data_to_write, range_name='A1')
print("工作表1的資料已根據日期排序並更新。")

工作表1的資料已根據日期排序並更新。


In [300]:
df['日期'] = pd.to_datetime(df['日期'])
df['月份'] = df['日期'].dt.month
df

,日期,分類,品項,金額,付款人,備註,月份
0,2026-01-01,飲食,午餐,120,范子昊,0,1
1,2026-02-03,交通,公車,12,范子昊,0,2
2,2026-03-01,飲食,早餐,100,范子昊,0,3
3,2026-03-07,飲食,午餐,120.0,范子昊,0,3


In [301]:
df['金額'] = pd.to_numeric(df['金額'], errors='coerce').fillna(0).astype(int)
monthly_total_spending = df.groupby('月份')['金額'].sum().reset_index()
print("每月總花費:")
print(monthly_total_spending)

每月總花費:
   月份   金額
0   1  120
1   2   12
2   3  220


**Reasoning**:
To further refine the analysis, I will calculate the total spending for each category within each month by grouping the DataFrame by '月份' and '分類' and summing the '金額'.



In [302]:
monthly_category_spending = df.groupby(['月份', '分類'])['金額'].sum().reset_index()
print("每月各分類總花費:")
print(monthly_category_spending)

每月各分類總花費:
   月份  分類   金額
0   1  飲食  120
1   2  交通   12
2   3  飲食  220


In [303]:
try:
    worksheet2 = gsheets.worksheet('工作表2')
except gspread.exceptions.WorksheetNotFound:
    worksheet2 = gsheets.add_worksheet(title='工作表2', rows="100", cols="20")

# Clear existing data in '工作表2'
worksheet2.clear()
print("'工作表2' 準備就緒，已清除原有數據。")

'工作表2' 準備就緒，已清除原有數據。


In [304]:
monthly_total_spending_list = [monthly_total_spending.columns.values.tolist()] + monthly_total_spending.values.tolist()
worksheet2.update('A1', monthly_total_spending_list)
print("每月總花費數據已寫入 '工作表2'。")

每月總花費數據已寫入 '工作表2'。


/tmp/ipykernel_204/3717924472.py:2: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  worksheet2.update('A1', monthly_total_spending_list)


In [305]:
monthly_total_spending_list = [monthly_total_spending.columns.values.tolist()] + monthly_total_spending.values.tolist()
worksheet2.update(values=monthly_total_spending_list, range_name='A1')
print("每月總花費數據已寫入 '工作表2'。")

# Calculate the starting row for the next section
next_row = len(monthly_total_spending_list) + 2 # +1 for blank line, +1 for title

# Write a title for monthly category spending
category_title_range = f'A{next_row}'
worksheet2.update(values=[['每月各分類總花費:']], range_name=category_title_range)
print("每月各分類總花費標題已寫入 '工作表2'。")

# Write monthly category spending data
monthly_category_spending_list = [monthly_category_spending.columns.values.tolist()] + monthly_category_spending.values.tolist()
category_data_range = f'A{next_row + 1}'
worksheet2.update(values=monthly_category_spending_list, range_name=category_data_range)
print("每月各分類總花費數據已寫入 '工作表2'。")

每月總花費數據已寫入 '工作表2'。
每月各分類總花費標題已寫入 '工作表2'。
每月各分類總花費數據已寫入 '工作表2'。
